In [1]:
!pip install -q transformers datasets evaluate seqeval matplotlib seaborn scikit-learn pytorch-crf

In [2]:
import os
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModel,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from torchcrf import CRF
from IPython.display import display

# ==========================================
# 0. CONFIGURATION - CHỈ CHỈNH SỬA Ở ĐÂY
# ==========================================
CHOSEN_MODEL = "bert-base-cased"
DATASET_FILE = "/content/dataset.jsonl"

# LỰA CHỌN CHẾ ĐỘ CHẠY (RUN_MODE)
# Gõ "ALL" để chạy đua top toàn bộ 6 cấu trúc.
# Hoặc copy 1 trong 6 tên sau để chạy riêng lẻ:
# "Base_CE", "FocalLoss", "DiceLoss", "CE+DiceLoss", "CRF", "CRF+FocalLoss"
RUN_MODE = "ALL"

print(f"🌟 CHUẨN BỊ TRAINING CHO MODEL: {CHOSEN_MODEL.upper()} 🌟")
print(f"🎯 CHẾ ĐỘ CHẠY: {RUN_MODE}")
print("="*70)

# ==========================================
# 1. LOAD & PREPARE DATA
# ==========================================
dataset = load_dataset("json", data_files=DATASET_FILE, split="train")
dataset = dataset.train_test_split(test_size=0.2, seed=42)

all_tags = [tag for tags_list in dataset["train"]["ner_tags"] for tag in tags_list]
unique_tags = sorted(list(set(all_tags)))

label2id = {label: i for i, label in enumerate(unique_tags)}
id2label = {i: label for i, label in enumerate(unique_tags)}

def encode_tags(example):
    example["ner_tags_id"] = [label2id.get(tag, 0) for tag in example["ner_tags"]]
    return example

dataset = dataset.map(encode_tags)

# Tính toán Class Weights dùng cho Focal Loss
tag_counts = pd.Series(all_tags).value_counts()
total_tags = len(all_tags)
class_weights = [total_tags / (len(unique_tags) * tag_counts[label]) for label in unique_tags]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

# ==========================================
# 2. DEFINING CUSTOM LOSSES & TRAINERS
# ==========================================
# --- FOCAL LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss_raw = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss_raw)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss_raw
        if self.weight is not None:
            alpha = self.weight[targets]
            focal_loss = focal_loss * alpha
        return focal_loss.mean()

class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        active_loss = inputs["attention_mask"].view(-1) == 1
        active_logits = logits.view(-1, model.config.num_labels)
        active_labels = torch.where(active_loss, labels.view(-1), torch.tensor(-100, device=labels.device))

        valid_indices = active_labels != -100
        valid_logits = active_logits[valid_indices]
        valid_labels = active_labels[valid_indices]

        focal_loss_fn = FocalLoss(weight=class_weights_tensor.to(model.device), gamma=2.0)
        loss = focal_loss_fn(valid_logits, valid_labels)
        return (loss, outputs) if return_outputs else loss

# --- DICE LOSS ---
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6, ignore_index=-100):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        num_classes = logits.shape[-1]
        probs = F.softmax(logits, dim=-1).view(-1, num_classes)
        targets = targets.view(-1)

        valid_mask = targets != self.ignore_index
        valid_targets = targets[valid_mask]
        valid_probs = probs[valid_mask]

        if valid_targets.numel() == 0:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        targets_one_hot = F.one_hot(valid_targets, num_classes=num_classes).float()
        intersection = torch.sum(valid_probs * targets_one_hot, dim=0)
        cardinality = torch.sum(valid_probs + targets_one_hot, dim=0)

        dice_score = (2. * intersection + self.smooth) / (cardinality + self.smooth)
        return (1. - dice_score).mean()

class DiceLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        dice_loss_fn = DiceLoss(ignore_index=-100)
        loss = dice_loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

# --- CE + DICE LOSS TRAINER ---
class CEDiceLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)

        ce_loss = outputs.loss
        logits = outputs.logits

        dice_loss_fn = DiceLoss(ignore_index=-100)
        dice_loss = dice_loss_fn(logits, labels)

        total_loss = ce_loss + dice_loss
        return (total_loss, outputs) if return_outputs else total_loss

# ==========================================
# 3. CRF ARCHITECTURE (WITH OPTIONAL FOCAL)
# ==========================================
class TransformerCRF(nn.Module):
    def __init__(self, model_name, num_labels, use_focal_loss=False, gamma=2.0):
        super(TransformerCRF, self).__init__()
        self.num_labels = num_labels
        self.use_focal_loss = use_focal_loss
        self.gamma = gamma
        self.config = AutoConfig.from_pretrained(model_name)
        self.transformer = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.classifier(outputs.last_hidden_state)

        loss = None
        if labels is not None:
            mask = attention_mask.bool()
            safe_labels = torch.where(labels == -100, torch.tensor(0, device=labels.device), labels)

            if self.use_focal_loss:
                nll = -self.crf(logits, tags=safe_labels, mask=mask, reduction='none')
                pt = torch.exp(-nll)
                loss = (((1 - pt) ** self.gamma) * nll).mean()
            else:
                loss = -self.crf(logits, tags=safe_labels, mask=mask, reduction='mean')

        eval_mask = attention_mask.bool()
        decoded_tags = self.crf.decode(logits, mask=eval_mask)

        fake_logits = torch.zeros_like(logits)
        for i, tags in enumerate(decoded_tags):
            for j, tag in enumerate(tags):
                fake_logits[i, j, tag] = 1.0

        return {"loss": loss, "logits": fake_logits} if loss is not None else {"logits": fake_logits}

# ==========================================
# 4. EVALUATION METRICS
# ==========================================
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# ==========================================
# 5. MODEL EXPORT UTILITY
# ==========================================
def export_model(model, tokenizer, arch_type, save_dir, id2label, label2id):
    os.makedirs(save_dir, exist_ok=True)
    print(f"\n💾 Đang tiến hành lưu model ({arch_type}) tại: {save_dir} ...")

    # 1. Lưu Tokenizer - Ép lưu thêm vocab.txt để đảm bảo đủ file cho backend
    tokenizer.save_pretrained(save_dir)
    # Một số tokenizer cần lệnh này để nhả file vocab.txt
    if hasattr(tokenizer, "save_vocabulary"):
        tokenizer.save_vocabulary(save_dir)

    # 2. Xử lý Config (Đảm bảo id2label luôn nằm trong config.json)
    if hasattr(model, "config"):
        model.config.id2label = id2label
        model.config.label2id = label2id
        model.config.save_pretrained(save_dir)

    # 3. Phân nhánh cách lưu trọng số
    if arch_type in ["CRF", "CRF+FocalLoss"]:
        # Lưu Custom PyTorch Model
        # Nên dùng đuôi .bin hoặc .pth (Backend PyTorch sẽ load file này)
        torch.save(model.state_dict(), os.path.join(save_dir, "pytorch_model.bin"))

        # Lưu thêm một file json chứa mapping nhãn riêng cho chắc chắn
        import json
        with open(os.path.join(save_dir, "labels.json"), "w") as f:
            json.dump({"id2label": id2label, "label2id": label2id}, f)

        print("✅ Đã lưu: Tokenizer (Full) + CRF Weights (.bin) + Config (.json) + Labels (.json)")
    else:
        # Lưu Standard Hugging Face Model
        # Ép lưu ở dạng safe_serialization=False nếu bạn muốn file .bin truyền thống
        model.save_pretrained(save_dir, safe_serialization=True)
        print("✅ Đã lưu: Tokenizer (Full) + HF Model (.safetensors) + Config (.json)")

    # 4. Kiểm tra sự tồn tại của file cốt lõi
    files = os.listdir(save_dir)
    print(f"📦 Các file hiện có trong folder: {files}")

# ==========================================
# 6. CORE TRAINING ENGINE
# ==========================================
def run_experiment(model_name, arch_type):
    base_name = model_name.split('-')[0].upper()
    arch_name = f"{base_name} + {arch_type}"
    print(f"\n🚀 Đang chạy: {arch_name}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name, add_prefix_space=True if "roberta" in model_name else False)

    def tokenize_and_align_labels(examples):
        tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
        labels = []
        for i, label in enumerate(examples["ner_tags_id"]):
            word_ids = tokenized_inputs.word_ids(batch_index=i)
            previous_word_idx = None
            label_ids = []
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    label_ids.append(label[word_idx])
                else:
                    label_ids.append(-100)
                previous_word_idx = word_idx
            labels.append(label_ids)
        tokenized_inputs["labels"] = labels
        return tokenized_inputs

    tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

    # ĐỊNH TUYẾN MÔ HÌNH VÀ TRAINER
    if arch_type == "Base_CE":
        model = AutoModelForTokenClassification.from_pretrained(
            model_name, num_labels=len(unique_tags), id2label=id2label, label2id=label2id
        )
        trainer_class = Trainer
    elif arch_type == "FocalLoss":
        model = AutoModelForTokenClassification.from_pretrained(
            model_name, num_labels=len(unique_tags), id2label=id2label, label2id=label2id
        )
        trainer_class = FocalLossTrainer
    elif arch_type == "DiceLoss":
        model = AutoModelForTokenClassification.from_pretrained(
            model_name, num_labels=len(unique_tags), id2label=id2label, label2id=label2id
        )
        trainer_class = DiceLossTrainer
    elif arch_type == "CE+DiceLoss":
        model = AutoModelForTokenClassification.from_pretrained(
            model_name, num_labels=len(unique_tags), id2label=id2label, label2id=label2id
        )
        trainer_class = CEDiceLossTrainer
    elif arch_type == "CRF":
        model = TransformerCRF(model_name, num_labels=len(unique_tags), use_focal_loss=False)
        trainer_class = Trainer
    elif arch_type == "CRF+FocalLoss":
        model = TransformerCRF(model_name, num_labels=len(unique_tags), use_focal_loss=True, gamma=2.0)
        trainer_class = Trainer

    training_args = TrainingArguments(
        output_dir=f"./temp_{arch_name.replace(' ', '_').replace('+', '_')}",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none"
    )

    trainer = trainer_class(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_results = trainer.evaluate()
    f1_score = eval_results['eval_f1']
    print(f"✅ Xong {arch_name}! F1: {f1_score:.4f}")

    # GỌI HÀM EXPORT MODEL TẠI ĐÂY
    save_path = f"./exported_models/{CHOSEN_MODEL.replace('/', '_')}_{arch_type.replace('+', '_')}"
    export_model(
        model=model,
        tokenizer=tokenizer,
        arch_type=arch_type,
        save_dir=save_path,
        id2label=id2label,
        label2id=label2id
    )

    return {
        "Architecture": arch_type,
        "F1-Score": f1_score,
        "Precision": eval_results['eval_precision'],
        "Recall": eval_results['eval_recall'],
        "Accuracy": eval_results['eval_accuracy']
    }

# ==========================================
# 7. EXECUTION & DISPLAY
# ==========================================
VALID_ARCHS = ["Base_CE", "FocalLoss", "DiceLoss", "CE+DiceLoss", "CRF", "CRF+FocalLoss"]

if RUN_MODE == "ALL":
    architectures = VALID_ARCHS
elif RUN_MODE in VALID_ARCHS:
    architectures = [RUN_MODE]
else:
    raise ValueError(f"❌ RUN_MODE không hợp lệ! Vui lòng chọn 'ALL' hoặc một trong: {VALID_ARCHS}")

results_list = []

for arch in architectures:
    try:
        res = run_experiment(CHOSEN_MODEL, arch_type=arch)
        results_list.append(res)
    except Exception as e:
        print(f"❌ Lỗi khi chạy {CHOSEN_MODEL} + {arch}: {e}")

# Hiển thị bảng kết quả
print("\n" + "*"*70)
print(f"🏆 BÁO CÁO KẾT QUẢ CHO: {CHOSEN_MODEL.upper()} 🏆")
print("*"*70)

df_results = pd.DataFrame(results_list)

if not df_results.empty:
    df_results = df_results.sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
    display(df_results)

    best_arch = df_results.iloc[0]['Architecture']
    best_f1 = df_results.iloc[0]['F1-Score']

    if len(architectures) > 1:
        print(f"\n🎉 Cấu trúc ĐỈNH NHẤT cho model này là: {best_arch} (F1: {best_f1:.4f}) 🎉")
    else:
        print(f"\n✅ Hoàn thành test độc lập cấu trúc: {best_arch} (F1: {best_f1:.4f})")

🌟 CHUẨN BỊ TRAINING CHO MODEL: BERT-BASE-CASED 🌟
🎯 CHẾ ĐỘ CHẠY: ALL


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/8800 [00:00<?, ? examples/s]

Map:   0%|          | 0/2201 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,5.664766,1.783399,0.892688,0.904118,0.898366,0.971528
2,1.449808,1.580936,0.905966,0.912556,0.909249,0.973769
3,1.055924,1.548894,0.905446,0.916776,0.911076,0.974128
4,0.897684,1.616536,0.901013,0.919104,0.909968,0.973769
5,0.652645,1.613707,0.906763,0.916921,0.911814,0.974551


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LOC seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


✅ Xong BERT + CRF+FocalLoss! F1: 0.9118

💾 Đang tiến hành lưu model (CRF+FocalLoss) tại: ./exported_models/bert-base-cased_CRF_FocalLoss ...
✅ Đã lưu: Tokenizer (Full) + CRF Weights (.bin) + Config (.json) + Labels (.json)
📦 Các file hiện có trong folder: ['tokenizer.model', 'config.json', 'tokenizer.json', 'tokenizer_config.json', 'labels.json', 'pytorch_model.bin']

**********************************************************************
🏆 BÁO CÁO KẾT QUẢ CHO: BERT-BASE-CASED 🏆
**********************************************************************


,Architecture,F1-Score,Precision,Recall,Accuracy
0,CE+DiceLoss,0.920218,0.918285,0.922159,0.975925
1,Base_CE,0.917210,0.914030,0.920413,0.975544
2,CRF,0.913744,0.908163,0.919395,0.974741
3,CRF+FocalLoss,0.911814,0.906763,0.916921,0.974551
4,DiceLoss,0.895908,0.898332,0.893496,0.968569
5,FocalLoss,0.850250,0.812078,0.892187,0.958930



🎉 Cấu trúc ĐỈNH NHẤT cho model này là: CE+DiceLoss (F1: 0.9202) 🎉


In [3]:
!zip -r exported_models.zip /content/exported_models

  adding: content/exported_models/ (stored 0%)
  adding: content/exported_models/bert-base-cased_CRF_FocalLoss/ (stored 0%)
  adding: content/exported_models/bert-base-cased_CRF_FocalLoss/tokenizer.model (deflated 49%)
  adding: content/exported_models/bert-base-cased_CRF_FocalLoss/config.json (deflated 57%)
  adding: content/exported_models/bert-base-cased_CRF_FocalLoss/tokenizer.json (deflated 70%)
  adding: content/exported_models/bert-base-cased_CRF_FocalLoss/tokenizer_config.json (deflated 43%)
  adding: content/exported_models/bert-base-cased_CRF_FocalLoss/labels.json (deflated 53%)
  adding: content/exported_models/bert-base-cased_CRF_FocalLoss/pytorch_model.bin (deflated 7%)
  adding: content/exported_models/bert-base-cased_CE_DiceLoss/ (stored 0%)
  adding: content/exported_models/bert-base-cased_CE_DiceLoss/tokenizer.model (deflated 49%)
  adding: content/exported_models/bert-base-cased_CE_DiceLoss/model.safetensors (deflated 7%)
  adding: content/exported_models/bert-base-ca